<a href="https://colab.research.google.com/github/votaquangnhat/LifeMate-Baseline/blob/main/notebooks/train_and_infer_qwen35_thinking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LifeMate — Qwen3.5-4B Thinking training and inference
Git-backed workflow: this notebook clones `training_pipeline/` and only supplies configuration and stage commands. **Commit and push the new notebook and updated pipeline before opening it in Colab.**

**Pipeline:** teammate Gipformer ASR (`best.pt`) → frozen BGE schema/history ranking → Qwen3.5-4B with thinking enabled → JSON tool execution → submission. Only language LoRA adapters are trained; ASR, BGE and the base weights stay frozen.

This is a **fresh adapter on a pretrained Qwen3.5 base**, not training a model from scratch. Your Qwen3 adapter is incompatible. Later sessions can continue a Qwen3.5 best adapter. Use **L4 or A100** for BF16 LoRA; Step 7 checks actual memory and optimizer updates before training. A higher public score is not guaranteed.


## 1. Mount Drive and set paths
Keep the same run name to resume the **same** configuration; use a new name for a different recipe. Checkpoints and reports are written directly to Drive. Dataset, environments and SQLite cache stay on local Colab disk.

ASR can be supplied as a JSONL from **any architecture** (including Gipformer), or generated in a later cell from a merged Whisper-compatible checkpoint. Training needs neither. Set `ASR_MODE` and the relevant path before the public stages.

**Training across days:**
- First day: `TRAIN_START='fresh'`, `RUN_NAME='day1'`, `EPOCHS=2`.
- Next day: `TRAIN_START='best'`, `PREVIOUS_RUN_NAME='day1'`, `RUN_NAME='day2'`, `EPOCHS=2`.
- Following day: use `PREVIOUS_RUN_NAME='day2'` with another new run name.

`best` loads the previous selected LoRA weights and trains for **additional** epochs with a fresh optimizer and learning-rate schedule. It is a new fine-tuning session, not an exact continuation of a single 10-epoch schedule. The previous best also competes in validation selection, so a worse new epoch does not replace it. Keep the validation split fixed; reserve held-out test evaluation for the final session.

For an interrupted session, keep its `RUN_NAME`, start settings and epoch count unchanged; `RESUME_CHECKPOINT='auto'` restores the full latest checkpoint, including optimizer/scheduler. A completed run is not extended in place: start a new run from its best instead. For a new fresh start, use a new run name.

**Daily training:** the default is one epoch. After selection/export, tomorrow set `RUN_DATE` to the new date, `TRAIN_START='best'`, and `PREVIOUS_RUN_NAME` to today's exact name. This starts another session with a fresh optimizer from the selected adapter. For an interrupted session, keep the date, run name, epochs, config and Git commit unchanged; `RESUME_CHECKPOINT='auto'` restores optimizer progress. Use `_v2` for another run on the same date.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, subprocess
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/Zalo-AI-Challenge-2026')
MODEL_PROFILE = 'qwen35_thinking'
RUN_DATE = '20261109'  # YYYYMMDD when this run starts. Keep fixed when resuming.
RUN_NAME = f'qwen35_4b_thinking_{RUN_DATE}_v1'
EPOCHS = 1  # Additional epochs for this run when TRAIN_START='best'.
TRAIN_START = 'fresh'  # 'fresh': base model; 'best': previous run's selected adapter.
PREVIOUS_MODEL_PROFILE = 'qwen35_thinking'
RUN_DATE = '20261109'  # YYYYMMDD when this run starts. Keep fixed when resuming.
RUN_NAME = f'qwen35_4b_thinking_{RUN_DATE}_v1'  # For 'best'; use a NEW RUN_NAME above.
RUN_BASELINE = False  # Optional untrained planner score on the SAME validation split.
ASR_MODE = 'transcripts'  # 'transcripts' or 'whisper'
ASR_TRANSCRIPTS = DRIVE_ROOT / 'inputs/public_asr.jsonl'
ASR_MODEL = ''  # Only for whisper: teammate's merged checkpoint folder / Hub ID.
EXPORT_BASE_WEIGHTS = True  # Self-contained best_model bundle; needs several GB on Drive.
RESUME_CHECKPOINT = 'auto'  # Resume an interruption within this SAME run, including optimizer/scheduler.

WORK_ROOT = Path('/content/lifemate_qwen35_train')
OUTPUT_DIR = DRIVE_ROOT / 'runs' / RUN_NAME
for folder in (DRIVE_ROOT / 'inputs', WORK_ROOT, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = OUTPUT_DIR / 'run_config.json'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Persistent run:', OUTPUT_DIR)


### Clone the implementation
Push the new files before the first Colab run. To receive later updates, explicitly refresh the checkout in a new runtime; this cell does not automatically pull changes into an active training session.


In [ ]:
# Clone once. Existing checkouts are retained so reruns do not silently change code.
REPO_URL = 'https://github.com/votaquangnhat/LifeMate-Baseline.git'
REPO_REF = 'main'  # Use a commit SHA for reproducible runs.
PROJECT = Path('/content/lifemate-qwen35-training')
if not PROJECT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'checkout', REPO_REF], check=True)
CODE_DIR = PROJECT / 'training_pipeline'
assert (CODE_DIR / 'colab_workflow.py').is_file(), 'Push training_pipeline/ to GitHub first, then use a fresh clone containing those files.'
revision = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
expected = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', REPO_REF], text=True).strip()
assert revision == expected, 'Existing checkout differs from REPO_REF; explicitly select the intended revision before continuing.'
revision_file = OUTPUT_DIR / 'repository_revision.txt'
if revision_file.exists():
    assert revision_file.read_text().strip() == revision, 'Repository revision changed: use a new RUN_NAME.'
revision_file.write_text(revision + '\n')
sys.path.insert(0, str(CODE_DIR))
import colab_workflow as workflow
print('Repository:', PROJECT, 'commit:', revision)


## 2. Download and extract the same dataset
The Drive file ID below comes from your baseline. Alternatively put the ZIP at `LOCAL_ARCHIVE` before running. Data is extracted locally; it is not included in the model export.


In [ ]:
DATA_ROOT = workflow.prepare_data(WORK_ROOT)


## 3. Install an isolated agent runtime
Preserves Colab CUDA PyTorch and displays GPU name/memory. Qwen3.5 requires Transformers 5; each GPU stage runs in a separate process. Package versions and Git/model revisions are recorded for resume.

This recipe uses **unquantized BF16 LoRA**, following [Unsloth's Qwen3.5 guidance](https://unsloth.ai/docs/models/qwen3.5/fine-tune). Model: `unsloth/Qwen3.5-4B`; official `Qwen/Qwen3.5-4B` tokenizer. Thinking is explicitly enabled, not inferred from a model name.


In [ ]:
AGENT_PYTHON = workflow.setup_environment(WORK_ROOT, OUTPUT_DIR, MODEL_PROFILE=MODEL_PROFILE)


## 4. Training configuration
The BGE reranker retains short schemas for **all** installed tools, prioritizing relevant descriptions; it keeps recent history and retrieves relevant older blocks. No gold action is used to choose input schemas/history. Prompts, targets and current tool observations are not silently truncated. Preparation measures actual lengths; preflight checks the resulting memory demand.

The split is grouped by shared conversations/audio (85/10/5), matching the completed teammate recipe. Share `split_assignments.json` with your ASR teammate if you want an uncontaminated end-to-end held-out comparison. Gold-transcript scores measure only the planner.

Qwen3.5 settings: rank 16 / alpha 32, learning rate 1e-4, batch 1 × accumulation 8, BF16 backbone, FP32 adapter parameters, AdamW. Up to 8192 total tokens; inference allows 512 thinking tokens and reserves 512 tokens for the JSON action. A token-budget processor closes thinking before it consumes the action budget.


In [ ]:
CONFIG = workflow.configure(
    MODEL_PROFILE=MODEL_PROFILE,
    DATA_ROOT=DATA_ROOT, OUTPUT_DIR=OUTPUT_DIR, CODE_DIR=CODE_DIR, CONFIG_PATH=CONFIG_PATH,
    WORK_ROOT=WORK_ROOT, DRIVE_ROOT=DRIVE_ROOT, RUN_NAME=RUN_NAME,
    TRAIN_START=TRAIN_START, PREVIOUS_RUN_NAME=PREVIOUS_RUN_NAME,
    EPOCHS=EPOCHS, RESUME_CHECKPOINT=RESUME_CHECKPOINT, ASR_TRANSCRIPTS=ASR_TRANSCRIPTS,
)


## 5. Check the repository runtime
Run the CPU contract tests before loading model weights. A source snapshot is saved to the run folder for provenance; execution uses the cloned `training_pipeline/` files. The notebook does not write Python implementations.


In [ ]:
workflow.check_runtime(AGENT_PYTHON, CODE_DIR, OUTPUT_DIR)

def run_stage(stage):
    workflow.run_stage(AGENT_PYTHON, CODE_DIR, CONFIG_PATH, stage)


## 6. Prepare SFT and evaluate schema coverage
This replays labeled **training** trajectories to obtain real observations and reference hashes. It produces assistant-only targets for each next action and `done`; user text, schemas, history and tool results are masked from loss. All train/dev/test data stays separate. Nothing from public labels or environment files is read by the agent.

Inspect `retrieval_report.json` and `length_report.json`. Schema coverage must be 100%; top-k description recall is a separate diagnostic. Cached preparation is reused only if code, inputs, tokenizer, packages and configuration match.

**Thinking supervision:** the labels contain no expert reasoning traces. The pipeline adds short, rule-based procedural text before `</think>`, followed by the gold JSON action. This is a simple scaffold, not distilled reasoning from your teammate's model. Only the current action and preceding visible observations are available; future results are not input. Both scaffold and action receive loss.


In [ ]:
run_stage('prepare')
CONFIG = json.loads(CONFIG_PATH.read_text())
for name in ('data_summary.json', 'length_report.json', 'retrieval_report.json'):
    print(name, (OUTPUT_DIR / name).read_text())


## 7. GPU preflight
Five real optimizer updates test representative and longest sequences, finite loss and peak VRAM. Qwen3.5 uses BF16 on L4/A100; T4 is not supported by this recipe. Preflight weights are discarded. If memory is insufficient, use a larger GPU; sequences are not silently truncated.


In [ ]:
run_stage('smoke')
print((OUTPUT_DIR / 'preflight_report.json').read_text())


## 8. Optional baseline, then train
Training saves full resume checkpoints every 100 optimizer steps and at each epoch boundary directly to Drive. `auto` resumes the latest complete checkpoint after a disconnect. If the notebook is interrupted, reconnect, rerun the setup/preparation cells, and continue. A GPU/packaging mismatch is reported rather than silently changing the run.

Selection evaluates epoch checkpoints only; periodic checkpoints are for resuming. Disk usage can grow because optimizer states are retained.

When `TRAIN_START="best"`, both preflight and actual training start from the snapshotted previous best adapter. Its weight hash is part of the training signature; full interrupted-session checkpoints take precedence when resuming.


In [ ]:
if RUN_BASELINE:
    run_stage('baseline')
    print((OUTPUT_DIR / 'baseline_metrics.json').read_text())
run_stage('train')
print((OUTPUT_DIR / 'training_complete.json').read_text())


## 9. Select the best epoch by full validation rollout
Every validation sample runs through ToolEnv, including the model’s own intermediate decisions. Selection uses the teammate’s reconstruction of the competition metric (state-hash gate and one-to-one tool-call matching), not teacher-forced loss. The source field `official_score_100` is a local reconstruction, not an official leaderboard result. Ties choose the earliest checkpoint. Internal test scores do not select the model.

In `best` mode, the incoming previous best is evaluated on the same validation split alongside the new epoch checkpoints. Ties retain the previous best.


In [ ]:
run_stage('select')
print((OUTPUT_DIR / 'selection.json').read_text())


## 10. Save the best model to Drive now
`best_model/adapter/` holds the selected trained weights and tokenizer. By default this cell also saves the exact BF16 base and BGE weights, so the bundle contains the required model weights as well as runtime source/configuration. This is a **base + LoRA bundle**, not merged full-precision weights. The model and adapter must be loaded together using the supplied runtime. If `EXPORT_BASE_WEIGHTS=False`, the pinned Hub models remain required at inference.

The saved `deployment_config.json` points to the exported bundle on this Drive. For another machine, update paths. ASR is supplied separately. Model export does not include competition data or ASR weights.


In [ ]:
BEST_MODEL = workflow.export_model(
    AGENT_PYTHON, OUTPUT_DIR, CODE_DIR, CONFIG_PATH,
    EXPORT_BASE_WEIGHTS=EXPORT_BASE_WEIGHTS,
)


## 11. Held-out planner evaluation
Run after model selection. This uses gold transcripts and does not measure ASR quality. `planner_seconds` excludes ASR and reranking; it does not certify the end-to-end 30-second requirement or final Docker deployment. Review aborted rollouts and tool errors in the saved traces.

For daily continued training, skip this cell until your final session; use validation alone for intermediate decisions.


In [ ]:
run_stage('test')
print((OUTPUT_DIR / 'test_metrics.json').read_text())


## 12. Optional teammate ASR handoff — skip to Step 13 to infer now
Your teammate can produce `inputs/public_asr.jsonl` with one record per public ID:

```json
{"id":"public_0001","audio_text":"recognized final request","history_audio_texts":["first user history turn","second user history turn"]}
```

`history_audio_texts` contains **only user turns**, in their original order; use `[]` for no user history. A dictionary keyed by exact relative audio paths is also accepted. Blank/missing/duplicate records fail before any public environment is created. Their ASR may do its own Vietnamese ITN; preserve names and quoted message text. This notebook does not blindly rewrite numbers inside free-text messages.

The first cell exports a manifest of audio paths for the teammate. The second either validates their file or runs inference from a merged Whisper-compatible checkpoint. For Gipformer or another architecture, use transcript mode; do not pass its checkpoint to the Whisper loader.


In [ ]:
workflow.write_asr_manifest(DATA_ROOT, DRIVE_ROOT, ASR_TRANSCRIPTS)


In [ ]:
workflow.prepare_asr(
    AGENT_PYTHON, CODE_DIR, DATA_ROOT, CONFIG_PATH,
    ASR_MODE, ASR_TRANSCRIPTS, ASR_MODEL,
)


## 13. Independent inference with your trained Gipformer ASR
Run the next cell directly in a Colab GPU session; Steps 1–12 are not required again.

**Before running:** upload only your teammate's `best.pt` to `/content/drive/MyDrive/Zalo-AI-Challenge-2026/ASR-cp/best.pt`. Step 13 is already configured for this exact path, matching the teammate notebook's direct `best.pt` loading. No `best_export` folder is required. The loader reads configuration embedded in the checkpoint and automatically downloads missing release tokenizer/vocabulary/architecture assets from the pinned revision. The trained ASR weights always come from your `best.pt`.

Default pipeline: **fine-tuned Gipformer 1.5 → BGE/history compiler → your selected Qwen3 adapter → ToolEnv → submission**. ASR includes all user-history audio. Decoder selection/normalization comes from checkpoint metadata. Only supplied context and prior history can bias decoding; hidden environment state is not read. ASR has its own Python 3.11 / PyTorch 2.8 CUDA 12.8 environment, separate from the planner. First installation can take several minutes.

`INFERENCE_MAX_LENGTH=8192` increases the planner context for the previously observed 5,218-token overflow. It changes inference only; GPU memory/latency still depend on the runtime. **Early planner stops print diagnostics and are saved in `submission_quality_report.json`; they do not block export of a complete, schema-valid submission.** No calls or hashes are invented or removed. Missing records, invalid files and real setup failures still stop execution.

Set `TRAINED_RUN_NAME` to your completed planner run. Use a new `INFERENCE_NAME` when changing checkpoint, decoder or code. Training files remain unchanged; outputs live under `runs/<TRAINED_RUN_NAME>/inference/<INFERENCE_NAME>/`, and validated `submission.jsonl` is also copied to the Drive root. Completed inference records are reused on identical reruns; an interrupted in-flight environment is not reset.

`ASR_MODE='transcripts'` still accepts a teammate JSONL through `TRANSCRIPTS_PATH`; `ASR_MODE='whisper'` remains an explicit fallback. A better ASR does not guarantee a particular leaderboard score. CPU checks do not substitute for running the actual downloaded checkpoint on a Colab GPU.

The planner and thinking settings are restored from the saved Qwen3.5 run. Update `TRAINED_RUN_NAME` after each new training session. Early stopped rollouts print their errors and still export complete, schema-valid predictions.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import sys, subprocess, importlib

# Everything this cell needs is configured here; no earlier cells required.
TRAINED_RUN_NAME = 'qwen35_4b_thinking_20261109_v1'  # Match the completed training run.
INFERENCE_NAME = 'gipformer_qwen35_20261109_v1'  # New name when code/settings change.
ASR_MODE = 'gipformer'  # 'gipformer', 'whisper', or 'transcripts'.
ASR_CHECKPOINT = '/content/drive/MyDrive/Zalo-AI-Challenge-2026/ASR-cp/best.pt'
ASR_DECODER = 'auto'  # Saved decoder choice; fallback: beam_context with visible history only.
INFERENCE_MAX_LENGTH = 8192  # Inference-only; no planner retraining.
ASR_MODEL = ''  # Ignored in gipformer mode; only set for optional whisper mode.
TRANSCRIPTS_PATH = ''  # In transcript mode: absolute path to teammate JSONL.
REPO_URL = 'https://github.com/votaquangnhat/LifeMate-Baseline.git'
REPO_REF = 'main'
INFER_PROJECT = Path('/content/lifemate-inference')
if not INFER_PROJECT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(INFER_PROJECT)], check=True)
# Refresh code even when this Colab session previously cloned an older version.
subprocess.run(['git', '-C', str(INFER_PROJECT), 'fetch', 'origin', REPO_REF], check=True)
subprocess.run(['git', '-C', str(INFER_PROJECT), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
INFER_CODE = INFER_PROJECT / 'training_pipeline'
sys.path.insert(0, str(INFER_CODE))
# A previous training cell may have imported an older workflow module.
sys.modules.pop('colab_workflow', None)
sys.modules.pop('infer_standalone', None)
for name in list(sys.modules):
    if name == 'gipformer_asr' or name == 'gipformer' or name.startswith('gipformer.'):
        sys.modules.pop(name, None)
importlib.invalidate_caches()
import infer_standalone
SUBMISSION = infer_standalone.run(
    drive_root='/content/drive/MyDrive/Zalo-AI-Challenge-2026',
    trained_run_name=TRAINED_RUN_NAME, inference_name=INFERENCE_NAME, code_dir=INFER_CODE,
    asr_mode=ASR_MODE, asr_model=ASR_MODEL, transcripts_path=TRANSCRIPTS_PATH,
    asr_checkpoint=ASR_CHECKPOINT, asr_decoder=ASR_DECODER,
    inference_max_length=INFERENCE_MAX_LENGTH,
)
print('Ready to submit:', SUBMISSION)


In [ ]:
# Optional download after Step 13; the Drive copy already persists.
from google.colab import files
files.download(str(SUBMISSION))


## Implementation and verification
Training/runtime code and CPU tests live in `training_pipeline/`. Review details and known limits are in `notebooks/train_and_infer_review.md`.

Use `TRAIN_START='fresh'` with a new run name to start from the base model. Use `TRAIN_START='best'`, yesterday's `PREVIOUS_RUN_NAME`, and a new run name to train additional epochs. Full interrupted-session resume remains separate via `RESUME_CHECKPOINT='auto'`.

GPU training and public scoring still require the Colab run. The code refactor does not change the model recipe, preprocessing, or scoring.

The original Qwen3 notebook is preserved. This recipe changes the planner and precision but does not reproduce every detail of the teammate’s 56.79 run. Local CPU/tokenizer checks do not establish GPU memory use or leaderboard performance.
